<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/01-python-foundations/01_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1: Python Foundations

This module covers the core of the language: types, control flow, functions and comprehensions. If you've written Python before, some of this will look familiar, but read it anyway. A lot of the bugs I've seen in ML code come from things in this module, like comparing floats with `==`, mutable default arguments, and two variables accidentally pointing to the same list.

Everything here is pure Python. No NumPy yet. That's on purpose. Once you know what the plain Python version of something looks like, it's much easier to understand what NumPy and PyTorch are doing for you later.

**Contents**

1. Variables and how names work
2. Numbers (and why floats will bite you)
3. Strings and f-strings
4. Booleans, None and truthiness
5. Control flow
6. Functions
7. Comprehensions
8. Handling errors
9. Putting it together: fitting a line with gradient descent

Run the cells from top to bottom. Change things and run them again. That's how this stuff sticks.

## 1. Variables and how names work

In Python, a variable is a name that points to an object. The object has a type, the name doesn't. That's why you can do this:

In [1]:
x = 42
print(type(x))

x = "forty-two"
print(type(x))

<class 'int'>
<class 'str'>


This is called dynamic typing. It's convenient, but it also means Python won't stop you from passing a string where your function expected a number. You only find out when the code runs, sometimes an hour into a training run.

The more important thing to understand is that assignment never copies data. `b = a` makes `b` point to the same object as `a`. With numbers and strings you'll never notice, because they can't be changed in place. With lists you will:

In [2]:
features = [0.5, 1.2, 3.3]
scaled = features          # not a copy, just a second name for the same list
scaled[0] = 999

print(features)

[999, 1.2, 3.3]


We changed `scaled`, and `features` changed too, because they are the same list. If you want an independent copy, you have to ask for one:

In [3]:
features = [0.5, 1.2, 3.3]
scaled = features.copy()   # list(features) or features[:] also work
scaled[0] = 999

print("features:", features)
print("scaled:  ", scaled)

features: [0.5, 1.2, 3.3]
scaled:   [999, 1.2, 3.3]


You can check whether two names point to the same object with `is`:

In [4]:
a = [1, 2, 3]
b = a
c = a.copy()

print(a is b)   # same object
print(a is c)   # different object
print(a == c)   # but the contents are equal

True
False
True


`==` compares values. `is` compares identity, meaning "is this literally the same object in memory". Keep this in mind. The same issue comes back with NumPy array views in Module 6, and it causes some very confusing bugs.

## 2. Numbers

The two numeric types you'll use all the time are `int` and `float`.

In [5]:
print(7 / 2)    # true division, always gives a float
print(7 // 2)   # floor division
print(7 % 2)    # remainder
print(2 ** 10)  # power

3.5
3
1
1024


Python ints have no size limit, which is nice:

In [6]:
print(2 ** 200)

1606938044258990275541962092341162602522202993782792835301376


Floats are a different story. They're stored in binary with limited precision (64 bits), so most decimal numbers can't be represented exactly:

In [7]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

0.30000000000000004
False


This isn't a Python bug. Every language that uses standard IEEE 754 floats does the same thing. But it matters a lot in ML, where nearly every number is a float.

**Never compare floats with `==`.** Compare with a tolerance instead:

In [8]:
import math

print(math.isclose(0.1 + 0.2, 0.3))
print(abs((0.1 + 0.2) - 0.3) < 1e-9)

True
True


A practical example is early stopping, where you stop training once the loss stops improving. Checking `if new_loss == old_loss` would almost never be true. Instead you decide how small an improvement still counts:

In [9]:
old_loss = 0.48213
new_loss = 0.48209

min_delta = 1e-3
if old_loss - new_loss < min_delta:
    print("Improvement too small, stop training")

Improvement too small, stop training


There are two special float values you'll meet sooner or later: infinity and NaN ("not a number"). NaN in particular shows up when training goes wrong, for example when you take the log of zero or divide zero by zero.

In [10]:
inf = float("inf")
nan = float("nan")

print(inf > 1e308)
print(nan == nan)          # NaN isn't equal to anything, not even itself
print(math.isnan(nan))     # this is how you check for it

True
False
True


`nan == nan` being `False` is not a typo. If your loss suddenly prints `nan`, every comparison against it is false, which can quietly break logic like "keep track of the best loss so far". Always check with `math.isnan`.

You convert between types by calling the type like a function:

In [11]:
print(int(3.99))      # truncates, doesn't round
print(round(3.99))
print(float("2.5"))
print(int("42"))

3
4
2.5
42


Two things here surprise people at least once. `int()` truncates toward zero, so `int(-3.7)` is `-3`, not `-4`. And `round()` uses "banker's rounding", which rounds halves to the nearest even number:

In [12]:
print(int(-3.7), round(2.5), round(3.5))

-3 2 4


## 3. Strings

Strings are immutable sequences of characters. You'll use them for file paths, class labels and log messages, and if you get into NLP, as your actual data.

In [13]:
label = "Positive"

print(label.lower())
print(label.upper())
print(len(label))
print(label[0], label[-1])  # first and last character
print(label[:3])            # first 3 characters

positive
POSITIVE
8
P e
Pos


Slicing follows the pattern `[start:stop:step]`, and `stop` is not included. This exact syntax works on lists, tuples, NumPy arrays and PyTorch tensors too, so it's worth getting comfortable with it now.

In [14]:
s = "abcdefgh"
print(s[2:5])    # 'cde'
print(s[:4])     # 'abcd'
print(s[4:])     # 'efgh'
print(s[::2])    # every second character
print(s[::-1])   # reversed

cde
abcd
efgh
aceg
hgfedcba


Some string methods that come up all the time when cleaning text data:

In [15]:
raw = "   The Movie was GREAT!!  "

clean = raw.strip().lower()
print(repr(clean))

words = clean.split()
print(words)

print(" ".join(words))
print(clean.replace("!", ""))
print(clean.startswith("the"))

'the movie was great!!'
['the', 'movie', 'was', 'great!!']
the movie was great!!
the movie was great
True


### f-strings

For formatting output, use f-strings. You'll print training progress constantly, and f-strings keep it readable:

In [16]:
epoch = 3
loss = 0.0456789
accuracy = 0.91234

print(f"epoch {epoch}: loss={loss:.4f}, acc={accuracy:.2%}")

epoch 3: loss=0.0457, acc=91.23%


The part after the colon is called a format spec. These are the ones I use most:

In [17]:
x = 1234.56789

print(f"{x:.2f}")       # 2 decimal places
print(f"{x:,.1f}")      # thousands separator
print(f"{x:e}")         # scientific notation
print(f"{0.0003:.1e}")  # handy for learning rates
print(f"{42:>8}")       # right-align in 8 characters
print(f"{42:05d}")      # pad with zeros

1234.57
1,234.6
1.234568e+03
3.0e-04
      42
00042


You can also put `=` inside the braces. It prints the expression and its value, which is great for quick debugging:

In [18]:
learning_rate = 3e-4
batch_size = 32
print(f"{learning_rate=}, {batch_size=}")

learning_rate=0.0003, batch_size=32


## 4. Booleans, None and truthiness

Comparisons give you `True` or `False`, and you combine them with `and`, `or` and `not`.

In [19]:
p = 0.73
print(p > 0.5)
print(p > 0.5 and p < 0.9)
print(0.5 < p < 0.9)   # Python lets you chain comparisons

True
True
True


The chained form `0.5 < p < 0.9` reads like math, and it's the normal way to write range checks in Python.

`None` is Python's way of saying "no value". It's often used as the default for optional arguments. Check for it with `is`, not `==`:

In [20]:
best_loss = None

if best_loss is None:
    print("No loss recorded yet")

No loss recorded yet


### Truthiness

Any object can be used as a condition. Empty containers, zero and `None` count as false. Almost everything else counts as true.

In [21]:
for value in [0, 0.0, "", [], {}, None, 1, -0.5, "a", [0]]:
    print(f"{value!r:>6} -> {bool(value)}")

     0 -> False
   0.0 -> False
    '' -> False
    [] -> False
    {} -> False
  None -> False
     1 -> True
  -0.5 -> True
   'a' -> True
   [0] -> True


This lets you write `if items:` instead of `if len(items) > 0:`. But be careful with numbers. If zero is a legitimate value, `if value:` will treat it as missing:

In [22]:
def describe(dropout=None):
    if dropout:                      # bug: 0.0 is a valid dropout rate
        return f"dropout={dropout}"
    return "no dropout set"

print(describe(0.0))


def describe_fixed(dropout=None):
    if dropout is not None:
        return f"dropout={dropout}"
    return "no dropout set"

print(describe_fixed(0.0))

no dropout set
dropout=0.0


The first version claims no dropout was set, even though we explicitly set it to 0.0. When `None` means "not provided", always test with `is None` or `is not None`.

## 5. Control flow

### if / elif / else

In [23]:
def grade_model(accuracy):
    if accuracy >= 0.95:
        return "excellent"
    elif accuracy >= 0.85:
        return "good"
    elif accuracy >= 0.70:
        return "okay"
    else:
        return "needs work"

for acc in [0.97, 0.88, 0.72, 0.4]:
    print(acc, grade_model(acc))

0.97 excellent
0.88 good
0.72 okay
0.4 needs work


Python checks the conditions from top to bottom and runs the first one that's true, so the order matters. If you put the `>= 0.70` check first, every model would be "okay".

For a simple either/or value there's a one-line form:

In [24]:
prob = 0.62
prediction = 1 if prob >= 0.5 else 0
print(prediction)

1


### for loops

A `for` loop goes over the items of anything iterable: lists, strings, ranges, dictionaries, files.

In [25]:
for i in range(3):
    print("epoch", i)

epoch 0
epoch 1
epoch 2


`range(start, stop, step)` works like slicing, so `stop` is excluded:

In [26]:
print(list(range(5)))
print(list(range(2, 10, 3)))
print(list(range(5, 0, -1)))

[0, 1, 2, 3, 4]
[2, 5, 8]
[5, 4, 3, 2, 1]


When you need both the position and the item, use `enumerate` instead of `range(len(...))`:

In [27]:
losses = [0.9, 0.6, 0.45, 0.41]

# works, but clumsy
for i in range(len(losses)):
    print(i, losses[i])

print()

# better
for epoch, loss in enumerate(losses, start=1):
    print(f"epoch {epoch}: {loss}")

0 0.9
1 0.6
2 0.45
3 0.41

epoch 1: 0.9
epoch 2: 0.6
epoch 3: 0.45
epoch 4: 0.41


To loop over two lists side by side, use `zip`. This comes up constantly in ML: features with labels, predictions with true values, and so on.

In [28]:
y_true = [1, 0, 1, 1, 0]
y_pred = [1, 0, 0, 1, 1]

correct = 0
for t, p in zip(y_true, y_pred):
    if t == p:
        correct += 1

print(f"accuracy: {correct / len(y_true):.2f}")

accuracy: 0.60


One thing to watch out for: `zip` stops at the shortest input and doesn't complain. If your two lists have different lengths, that's usually a bug, and `zip` will hide it. Since Python 3.10 you can pass `strict=True` to get an error instead.

In [29]:
a = [1, 2, 3]
b = [10, 20]
print(list(zip(a, b)))   # the 3 is silently dropped

try:
    list(zip(a, b, strict=True))
except ValueError as e:
    print("error:", e)

[(1, 10), (2, 20)]
error: zip() argument 2 is shorter than argument 1


### while loops, break and continue

A `while` loop keeps going as long as its condition is true. It's useful when you don't know in advance how many steps you need.

In [30]:
loss = 1.0
step = 0

while loss > 0.1:
    loss *= 0.7          # pretend each step cuts the loss by 30%
    step += 1

print(f"reached loss {loss:.4f} after {step} steps")

reached loss 0.0824 after 7 steps


`break` leaves the loop early, and `continue` jumps to the next iteration. Here's a small training loop that uses both: it skips epochs with a NaN loss and stops early once the loss stops improving.

In [31]:
losses = [0.9, 0.7, float("nan"), 0.5, 0.48, 0.479, 0.4789]
best = float("inf")
patience = 2
bad_epochs = 0

for epoch, loss in enumerate(losses):
    if math.isnan(loss):
        print(f"epoch {epoch}: nan loss, skipping")
        continue

    if best - loss > 0.01:
        best = loss
        bad_epochs = 0
    else:
        bad_epochs += 1

    print(f"epoch {epoch}: loss={loss}, best={best}")

    if bad_epochs >= patience:
        print("no improvement, stopping early")
        break

epoch 0: loss=0.9, best=0.9
epoch 1: loss=0.7, best=0.7
epoch 2: nan loss, skipping
epoch 3: loss=0.5, best=0.5
epoch 4: loss=0.48, best=0.48
epoch 5: loss=0.479, best=0.48
epoch 6: loss=0.4789, best=0.48
no improvement, stopping early


This is a simplified version of the early stopping logic you'll find in real training frameworks like Keras and PyTorch Lightning.

## 6. Functions

Functions are how you stop repeating yourself. A good function does one thing, has a clear name, and says what it expects.

In [32]:
def accuracy(y_true, y_pred):
    """Return the fraction of predictions that match the true labels."""
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    return correct / len(y_true)

print(accuracy([1, 0, 1, 1], [1, 1, 1, 1]))

0.75


The string right under the `def` line is a docstring. It shows up when you call `help(accuracy)` and in your editor's tooltips. Write one for anything that isn't completely obvious.

### Default and keyword arguments

In [33]:
def predict_label(prob, threshold=0.5):
    return 1 if prob >= threshold else 0

print(predict_label(0.6))
print(predict_label(0.6, threshold=0.7))

1
0


Passing arguments by name makes calls much easier to read, especially when a function takes several numbers. `train(model, 0.001, 32, 10)` tells you nothing. `train(model, lr=0.001, batch_size=32, epochs=10)` tells you everything.

### Returning several values

A function can return several values at once. Python packs them into a tuple, and you can unpack them on the other side:

In [34]:
def min_max(values):
    return min(values), max(values)

low, high = min_max([4, 8, 1, 9, 3])
print(low, high)

1 9


### The mutable default argument trap

This is the classic Python gotcha. Default values are created once, when the function is defined, not every time it's called. If the default is something mutable like a list, every call shares the same list:

In [35]:
def log_metric(value, history=[]):
    history.append(value)
    return history

print(log_metric(0.9))
print(log_metric(0.8))   # expected [0.8], got both values

[0.9]
[0.9, 0.8]


The fix is to use `None` as the default and create the list inside the function:

In [36]:
def log_metric(value, history=None):
    if history is None:
        history = []
    history.append(value)
    return history

print(log_metric(0.9))
print(log_metric(0.8))

[0.9]
[0.8]


### *args and **kwargs

`*args` collects any extra positional arguments into a tuple. `**kwargs` collects extra keyword arguments into a dictionary. You'll see these all over ML libraries, usually for passing configuration down through several layers of functions.

In [37]:
def mean(*values):
    return sum(values) / len(values)

print(mean(1, 2, 3, 4))


def build_model(name, **config):
    print(f"building {name}")
    for key, value in config.items():
        print(f"  {key} = {value}")

build_model("mlp", hidden_size=128, dropout=0.2, activation="relu")

2.5
building mlp
  hidden_size = 128
  dropout = 0.2
  activation = relu


`*` and `**` also work the other way around, to unpack a list or a dict into arguments:

In [38]:
config = {"hidden_size": 64, "dropout": 0.1}
build_model("small_mlp", **config)

building small_mlp
  hidden_size = 64
  dropout = 0.1


### Scope

Variables created inside a function only exist inside that function. A function can read variables from outside, but as soon as it assigns to a name, that name becomes local.

In [39]:
learning_rate = 0.01

def show_lr():
    print("inside:", learning_rate)   # reading a global works

def change_lr():
    learning_rate = 0.5               # this creates a new local variable
    print("inside:", learning_rate)

show_lr()
change_lr()
print("outside:", learning_rate)      # unchanged

inside: 0.01
inside: 0.5
outside: 0.01


As a rule, pass in what a function needs as arguments and return what it produces. Functions that depend on global variables are harder to test and harder to reuse.

### Functions are objects

You can store functions in variables, put them in lists and pass them to other functions. This is how ML libraries let you plug in your own loss function or metric.

In [40]:
def mse(y_true, y_pred):
    return sum((t - p) ** 2 for t, p in zip(y_true, y_pred)) / len(y_true)

def mae(y_true, y_pred):
    return sum(abs(t - p) for t, p in zip(y_true, y_pred)) / len(y_true)

def evaluate(y_true, y_pred, metric):
    return metric(y_true, y_pred)

y_true = [3.0, 5.0, 2.5]
y_pred = [2.5, 5.0, 4.0]

for metric in [mse, mae]:
    print(metric.__name__, evaluate(y_true, y_pred, metric))

mse 0.8333333333333334
mae 0.6666666666666666


For small throwaway functions there's `lambda`. The most common place you'll see it is as the `key` argument to `sorted`, `min` and `max`:

In [41]:
results = [("model_a", 0.81), ("model_b", 0.92), ("model_c", 0.77)]

best = max(results, key=lambda r: r[1])
print(best)

ranked = sorted(results, key=lambda r: r[1], reverse=True)
print(ranked)

('model_b', 0.92)
[('model_b', 0.92), ('model_a', 0.81), ('model_c', 0.77)]


If a lambda gets any longer than this, write a normal function instead.

## 7. Comprehensions

A comprehension builds a new list (or dict, or set) from an existing iterable in a single expression. They're everywhere in Python code, so you need to be able to read them quickly.

In [42]:
values = [1, 2, 3, 4, 5]

# loop version
squares = []
for v in values:
    squares.append(v ** 2)
print(squares)

# comprehension version
squares = [v ** 2 for v in values]
print(squares)

[1, 4, 9, 16, 25]
[1, 4, 9, 16, 25]


Read it as "give me `v ** 2` for each `v` in `values`". You can add a filter at the end:

In [43]:
scores = [0.2, 0.9, 0.55, 0.1, 0.75]

high = [s for s in scores if s > 0.5]
print(high)

[0.9, 0.55, 0.75]


If you want to transform each element differently depending on a condition, rather than filter, the `if/else` goes before the `for`:

In [44]:
labels = [1 if s >= 0.5 else 0 for s in scores]
print(labels)

[0, 1, 1, 0, 1]


This trips up a lot of people, so to repeat it: a filtering `if` goes at the end, and an `x if condition else y` expression goes at the front.

### Dict and set comprehensions

In [45]:
classes = ["cat", "dog", "bird"]

class_to_index = {name: i for i, name in enumerate(classes)}
index_to_class = {i: name for name, i in class_to_index.items()}

print(class_to_index)
print(index_to_class)

{'cat': 0, 'dog': 1, 'bird': 2}
{0: 'cat', 1: 'dog', 2: 'bird'}


You'll build this exact pair of dictionaries in pretty much every classification project. Models work with integers, but people want to see class names.

Set comprehensions use curly braces without a colon, and they drop duplicates automatically:

In [46]:
text = "the cat sat on the mat the end"
vocab = {word for word in text.split()}
print(sorted(vocab))

['cat', 'end', 'mat', 'on', 'sat', 'the']


### Nested comprehensions

You can have more than one `for`. They run in the same order as nested loops would. A common use is flattening a list of lists:

In [47]:
batches = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]
flat = [x for batch in batches for x in batch]
print(flat)

[1, 2, 3, 4, 5, 6, 7, 8, 9]


And here's a matrix transpose, turning rows into columns:

In [48]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
]

transposed = [[row[j] for row in matrix] for j in range(len(matrix[0]))]
print(transposed)

[[1, 4], [2, 5], [3, 6]]


Once a comprehension needs two `for`s and an `if`, it's often clearer to write a normal loop. Readability is more important than fitting things on one line.

### Generator expressions

Swap the square brackets for parentheses and you get a generator expression. It produces values one at a time instead of building the whole list in memory. When you pass it straight into a function like `sum`, `max` or `any`, you can even drop the extra parentheses:

In [49]:
values = range(1_000_000)

total = sum(v * v for v in values)    # no million-element list gets created
print(total)

print(any(v > 999_998 for v in values))

333332833333500000


True


Module 4 goes much deeper into generators, because they're the basis of how data loaders stream datasets that don't fit in memory.

## 8. Handling errors

When something goes wrong, Python raises an exception. You can catch it with `try` and `except`:

In [50]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return 0.0

print(safe_divide(10, 2))
print(safe_divide(10, 0))

5.0
0.0


Always catch a specific exception type. A bare `except:` catches everything, including typos in your own code and Ctrl+C, which makes bugs very hard to track down.

Just as important is raising errors yourself when a function gets input it can't handle. Failing loudly right away is much better than returning a wrong number that silently ruins your results:

In [51]:
def accuracy(y_true, y_pred):
    if len(y_true) != len(y_pred):
        raise ValueError(
            f"length mismatch: {len(y_true)} labels vs {len(y_pred)} predictions"
        )
    if not y_true:
        raise ValueError("cannot compute accuracy of empty lists")
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)

try:
    accuracy([1, 0, 1], [1, 0])
except ValueError as e:
    print("error:", e)

error: length mismatch: 3 labels vs 2 predictions


Also notice the `sum(t == p for ...)` trick. `True` counts as 1 and `False` as 0, so summing booleans counts how many are true. You'll see this a lot.

## 9. Putting it together: fitting a line with gradient descent

Let's use everything from this module to do some real machine learning in pure Python. We have data points that roughly follow a straight line, and we want to find the line `y = w * x + b` that fits them best.

"Best" here means the line with the smallest mean squared error (MSE) between its predictions and the real `y` values. Gradient descent finds it by starting with a guess and repeatedly nudging `w` and `b` in the direction that makes the error smaller.

For MSE, the gradients (how much the error changes when we change `w` or `b`) are:

$$\frac{\partial L}{\partial w} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)\, x_i \qquad \frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)$$

If the calculus is new to you, don't worry about it yet. The idea is simple: if our predictions are too high, the gradient is positive, and subtracting it lowers the predictions. Module 9 covers this properly.

First, some fake data where we know the right answer:

In [52]:
import random

random.seed(0)

# the true relationship is y = 2x + 1, plus some noise
xs = [i / 10 for i in range(50)]
ys = [2 * x + 1 + random.gauss(0, 0.3) for x in xs]

print(xs[:5])
print([round(y, 2) for y in ys[:5]])

[0.0, 0.1, 0.2, 0.3, 0.4]
[1.28, 0.78, 1.2, 1.71, 1.5]


Now the three pieces: making predictions, measuring the error, and computing the gradients.

In [53]:
def predict(xs, w, b):
    return [w * x + b for x in xs]


def mse(y_true, y_pred):
    return sum((p - t) ** 2 for t, p in zip(y_true, y_pred)) / len(y_true)


def gradients(xs, ys, w, b):
    n = len(xs)
    preds = predict(xs, w, b)
    errors = [p - t for p, t in zip(preds, ys)]
    dw = 2 / n * sum(e * x for e, x in zip(errors, xs))
    db = 2 / n * sum(errors)
    return dw, db

And the training loop:

In [54]:
w, b = 0.0, 0.0
lr = 0.05

for epoch in range(501):
    dw, db = gradients(xs, ys, w, b)
    w -= lr * dw
    b -= lr * db

    if epoch % 50 == 0:
        loss = mse(ys, predict(xs, w, b))
        print(f"epoch {epoch:3d}  loss={loss:.4f}  w={w:.3f}  b={b:.3f}")

epoch   0  loss=0.6998  w=1.875  b=0.591
epoch  50  loss=0.1100  w=2.071  b=0.822
epoch 100  loss=0.1089  w=2.056  b=0.869
epoch 150  loss=0.1088  w=2.052  b=0.884
epoch 200  loss=0.1088  w=2.051  b=0.888
epoch 250  loss=0.1088  w=2.050  b=0.889
epoch 300  loss=0.1088  w=2.050  b=0.890
epoch 350  loss=0.1088  w=2.050  b=0.890
epoch 400  loss=0.1088  w=2.050  b=0.890
epoch 450  loss=0.1088  w=2.050  b=0.890
epoch 500  loss=0.1088  w=2.050  b=0.890


The model started at `w=0, b=0` and ended up close to the true values `w=2, b=1`. It doesn't hit them exactly because of the noise we added, and that's expected. That's what fitting a model to real data looks like.

Try changing things and watch what happens:

- Set `lr = 0.2`. The updates overshoot, and the loss blows up instead of going down. This is what a learning rate that's too high looks like.
- Set `lr = 0.001`. It still works, just painfully slowly.
- In the data cell, change the noise from `0.3` to `2.0`.

Every neural network you'll ever train runs a loop like this one. The model is bigger and the gradients are computed for you, but the structure is the same: predict, measure the error, compute the gradients, update the parameters, repeat.

## Summary

- Names point to objects. Assignment doesn't copy, so use `.copy()` when you need an independent list.
- Don't compare floats with `==`. Use `math.isclose` or a tolerance, and check for NaN with `math.isnan`.
- f-strings with format specs (`{loss:.4f}`, `{acc:.2%}`, `{lr:.1e}`) are how you print training logs.
- Use `is None` when `None` means "not provided", especially when 0 is a valid value.
- Use `enumerate` and `zip` instead of `range(len(...))`. Remember that `zip` silently drops extra items.
- Never use a mutable object like a list as a default argument.
- Functions can be passed around like any other value. That's how pluggable losses and metrics work.
- In comprehensions, a filtering `if` goes at the end and `x if c else y` goes at the front.
- Raise clear errors on bad input instead of returning wrong results.

Next, open `02_exercises.ipynb` and work through the problems. Try not to look at the solutions until you've had a proper go at each one.